In [ ]:
import os
os.environ["GROQ_API_KEY"]=""

In [ ]:
from google.colab import files
upload = files.upload()

In [ ]:
!pip uninstall -y langchain langchain_community openai

In [ ]:
!pip install -q langchain langchain_community langchain-openai faiss-cpu pypdf sentence-transformers

In [ ]:
from langchain_community.document_loaders import PyPDFLoader
pdf_file = list(upload.keys())[0]
print(pdf_file)
loader = PyPDFLoader(pdf_file)
documents = loader.load()
print("Total Pages Loaded:", len(documents))

In [ ]:
print(documents[1].page_content)

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1500, chunk_overlap=500)
docs = text_splitter.split_documents(documents)
print("Total Chunks:", len(docs))

In [ ]:
from langchain_community.embeddings import HuggingFaceEmbeddings
# If you meant 'BAAI/bge-small-en-v1.5' or similar, you can use it here:
embeddings = HuggingFaceEmbeddings(model_name="BAAI/bge-small-en-v1.5")

In [ ]:
from langchain_community.vectorstores import FAISS
vectorstore = FAISS.from_documents(docs,embeddings)
print("Vector DB Created Successfully")
print(vectorstore)

In [ ]:
retriever = vectorstore.as_retriever(search_kwargs={"k":3})
print (retriever)

In [ ]:
print(vectorstore.index_to_docstore_id)

In [ ]:
docs_list = list(vectorstore.docstore._dict.values())
print(docs_list[0].page_content)

In [ ]:
for i in range(vectorstore.index.ntotal):
  vector= vectorstore.index.reconstruct(i)
  print(f"vector{i}:")
  print(vector)
  print("-"*50)

In [ ]:
from langchain_openai import ChatOpenAI
import os

llm = ChatOpenAI(model_name="llama-3.3-70b-versatile",
                 api_key=os.getenv("GROQ_API_KEY"),
                 base_url="https://api.groq.com/openai/v1"
                 )

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# Prompt template
prompt = ChatPromptTemplate.from_template(
    """Answer the question based only on the context below:

{context}

Question: {question}
"""
)

# Format retrieved docs
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

# RAG pipeline (modern way)
rag_chain = (
    {"context": retriever | format_docs, "question": lambda x: x}
    | prompt
    | llm
    | StrOutputParser()
)


query = input("Ask a question: ")

result = rag_chain.invoke(query)

print("\n Answer:\n")
print(result)

